# Chapter 6 · Running DFT in Practice — Colab notebook

Back to the chapter: <https://dongzhaohe321418-lab.github.io/materials-simulation-handbook/ch06-running-dft/>

We rehearse the full DFT workflow — build a structure, attach a calculator, relax it, sweep an equation of state, plot the result. Quantum ESPRESSO is not pip-installable, so we use ASE's built-in EMT calculator as a stand-in: the *workflow* is identical to a real DFT run, only the energy model is cheaper. A clearly marked block shows how to swap EMT for the real Espresso calculator.

This notebook is meant for **Google Colab** rather than the in-browser JupyterLite kernel, because it needs heavy packages (and, where noted, a GPU) that cannot run under Pyodide. Open it in Colab, and where the install cell mentions it, switch the runtime to a GPU via **Runtime -> Change runtime type -> GPU** before running the rest.


## Install

`ase` is the only dependency here and installs in a few seconds. No GPU is needed for this notebook — EMT is a cheap analytic potential. A real DFT run would need a separate Quantum ESPRESSO or VASP installation, which is not pip-installable and is out of scope for a Colab notebook.

In [ ]:
!pip install ase


## Build a crystal structure with ASE

We start exactly as a real DFT study would: construct the periodic cell. Here it is an aluminium FCC primitive cell. ASE's `bulk` builder knows the crystal structures of the elements.

In [ ]:
from ase.build import bulk
from ase.calculators.emt import EMT

al = bulk('Al', crystalstructure='fcc', a=4.05)
print(al)
print('volume per atom:', al.get_volume() / len(al), 'A^3')


## Attach a calculator and relax the cell

We attach the EMT calculator — the stand-in for a DFT engine — and relax the atomic positions and cell with a filter plus an optimiser. This mirrors a DFT geometry optimisation step for step.

In [ ]:
from ase.optimize import BFGS
from ase.filters import FrechetCellFilter

al.calc = EMT()
print('initial energy:', al.get_potential_energy(), 'eV')

relaxed = al.copy()
relaxed.calc = EMT()
opt = BFGS(FrechetCellFilter(relaxed), logfile=None)
opt.run(fmax=0.001)
print('relaxed energy:', relaxed.get_potential_energy(), 'eV')
print('relaxed lattice constant:', relaxed.cell.cellpar()[0], 'A')


## Sweep an equation of state

Scan the lattice constant, compute the energy at each volume, and fit a Birch-Murnaghan equation of state to recover the equilibrium volume and bulk modulus. This is one of the most common DFT validation tasks.

In [ ]:
import numpy as np

def equation_of_state(a_values):
    out = {'a': [], 'V': [], 'E': []}
    for a in a_values:
        atoms = bulk('Al', crystalstructure='fcc', a=a)
        atoms.calc = EMT()
        out['a'].append(a)
        out['V'].append(atoms.get_volume())
        out['E'].append(atoms.get_potential_energy())
    return {k: np.array(v) for k, v in out.items()}

a_grid = np.linspace(3.8, 4.3, 11)
eos = equation_of_state(a_grid)
for a, V, E in zip(eos['a'], eos['V'], eos['E']):
    print(f'a = {a:.3f} A   V = {V:7.3f} A^3   E = {E:8.4f} eV')


## Fit and plot the equation of state

ASE ships a Birch-Murnaghan fitter. The fitted minimum is the predicted equilibrium volume; the curvature is the bulk modulus.

In [ ]:
import matplotlib.pyplot as plt
from ase.eos import EquationOfState

eos_fit = EquationOfState(eos['V'], eos['E'])
V0, E0, B = eos_fit.fit()
print(f'equilibrium volume V0 = {V0:.3f} A^3')
print(f'equilibrium energy E0 = {E0:.4f} eV')
print(f'bulk modulus B = {B / 1e-3:.1f} meV/A^3')

fig, ax = plt.subplots(figsize=(6, 4))
eos_fit.plot(ax=ax)
ax.set_title('Aluminium equation of state (EMT stand-in)')
plt.show()


## Using real DFT instead of EMT

Everything above is the genuine DFT workflow; only the energy model is a stand-in. To run real density functional theory, install Quantum ESPRESSO separately (it is not pip-installable) and swap the EMT calculator for ASE's `Espresso` calculator. The rest of the notebook — `bulk`, `BFGS`, `EquationOfState` — is unchanged. The cell below is **not meant to run here**; it shows the one substitution.

In [ ]:
# ---- TO USE REAL DFT: replace `EMT()` with the block below ----
# Requires a working Quantum ESPRESSO install and pseudopotentials.
#
# from pathlib import Path
# from ase.calculators.espresso import Espresso, EspressoProfile
#
# profile = EspressoProfile(
#     command='pw.x',  # or 'mpirun -np 4 pw.x'
#     pseudo_dir=Path.home() / 'pseudo/SSSP_1.3.0_PBE_efficiency',
# )
# al.calc = Espresso(
#     profile=profile,
#     pseudopotentials={'Al': 'Al.pbe-n-kjpaw_psl.1.0.0.UPF'},
#     input_data={
#         'system': {'ecutwfc': 50, 'ecutrho': 400,
#                    'occupations': 'smearing', 'smearing': 'mv',
#                    'degauss': 0.01},
#         'electrons': {'conv_thr': 1e-9, 'mixing_beta': 0.4},
#     },
#     kpts=(8, 8, 8),
# )
# energy = al.get_potential_energy()  # now a genuine DFT energy
print('See comment above: swap EMT for Espresso for production DFT.')
